# Synthetic MuSiQue generation and dataset-wide redundancy audit

Generation targets structural acceptance. After generation, one dataset audit job
scans the entire accepted question catalog and uses Claude on bounded catalog and
cross-group comparison batches. It reports redundancy and concentration patterns;
it does not approve individual questions or silently remove them.

Use the capstone kernel and restart it after updating imported code. Both paid
launch cells are disabled by default. Generation and dataset auditing are separately
configurable. Reading saved data and reports makes no API calls.

Generation is bounded by the configured request/repair caps. Audit catalog groups
cover every fitting question; extra cross-group similarity comparisons have a
separate configurable budget. Coverage limits and errors are reported. There is no
automatic resume. No per-question semantic Claude review is run.


In [1]:
import html
import json
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from IPython.display import HTML, display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/mas_sae/data/synthetic").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")


sys.path.insert(0, str(ROOT / "src"))

RUN_TYPE = "sample"

from mas_sae.data.synthetic.batch import run_batch
from mas_sae.data.synthetic.generation import PROMPT_VERSION
from mas_sae.data.synthetic.dataset_audit import AUDIT_VERSION, audit_dataset
from mas_sae.data.synthetic.generation_cli import DEFAULT_MODEL_ID, make_client
from mas_sae.data.synthetic.hop_types import HOP_TYPES

MODEL_ID = DEFAULT_MODEL_ID
DATASET_AUDIT = True
AUDIT_MODEL_ID = MODEL_ID
# AUDIT_CONCURRENCY = 16
AUDIT_CONCURRENCY = 64
AUDIT_BATCH_SIZE = 100
AUDIT_MAX_PROMPT_CHARS = 100000
AUDIT_MAX_NEIGHBOR_BATCHES = 200
AUDIT_MAX_ATTEMPTS = 2
AUDIT_MAX_TOKENS = 8192
HOP_TYPES_TO_REVIEW = tuple(HOP_TYPES)


# CONCURRENCY = 64  # Benchmark 1, 4, 8, then 16 in separate runs.
# CONCURRENCY = 128  # Benchmark 1, 4, 8, then 16 in separate runs.
# CONCURRENCY = 256  # Benchmark 1, 4, 8, then 16 in separate runs.
CONCURRENCY = 256  # Benchmark 1, 4, 8, then 16 in separate runs.
# CONCURRENCY = 512  # Benchmark 1, 4, 8, then 16 in separate runs.


TARGET_PER_TYPE = 200
MAX_REQUESTS_PER_TYPE = 300


MAX_ATTEMPTS = 2
MAX_TOKENS = 6000
TEMPERATURE = None  # Omit temperature for Opus.
CONNECT_TIMEOUT = 10
READ_TIMEOUT = 300
SDK_MAX_ATTEMPTS = 3
SEED = 42  # Local sampling/shuffling only; not a model-generation seed.
ORIGINAL_FILES = (
    ROOT / "data/MuSiQue/clean/train.json",
    ROOT / "data/MuSiQue/clean/validation.json",
)
# RUN_DIR = ROOT / "data/MuSiQue/synthetic/runs" / "benchmarks" / (
#     datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
#     + "_review30_" + uuid4().hex[:8]
# )
# RUN_DIR = ROOT / f"data/MuSiQue/synthetic/runs" / RUN_TYPE / (
#     datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
#     + "_review30_" + uuid4().hex[:8]
# )
RUN_DIR = ROOT / f"data/MuSiQue/synthetic/runs" / RUN_TYPE / "20261030T173616Z_review30_8cfe1443"
print(f"Model: {MODEL_ID} | prompt: {PROMPT_VERSION}")
print(f"Concurrency: {CONCURRENCY}")
print(f"Target: {TARGET_PER_TYPE * len(HOP_TYPES_TO_REVIEW)} accepted records")
print(f"Maximum generation client calls: {MAX_REQUESTS_PER_TYPE * len(HOP_TYPES_TO_REVIEW) * MAX_ATTEMPTS}")
print(f"Output: {RUN_DIR}")

print(f"Dataset audit: {DATASET_AUDIT} | version: {AUDIT_VERSION}")
print(f"Audit model: {AUDIT_MODEL_ID} | concurrency: {AUDIT_CONCURRENCY}")
print(f"Maximum questions/group: {AUDIT_BATCH_SIZE}; extra neighbor-group cap: {AUDIT_MAX_NEIGHBOR_BATCHES}")


Model: us.anthropic.claude-opus-5 | prompt: v3
Concurrency: 256
Target: 1200 accepted records
Maximum generation client calls: 3600
Output: /home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/sample/20261030T173616Z_review30_8cfe1443
Dataset audit: True | version: redundancy-v1
Audit model: us.anthropic.claude-opus-5 | concurrency: 64
Maximum questions/group: 100; extra neighbor-group cap: 200


In [2]:
import inspect
print(inspect.signature(make_client))

(model_id: str, *, connect_timeout: float = 10, read_timeout: float = 300, sdk_max_attempts: int = 3)


## Shared batch runner — no API calls here

The notebook and CLI both call `synthetic.batch.run_batch`. Up to `CONCURRENCY` workers generate independently;
one coordinator validates, deduplicates, writes files, and updates quotas. Accepted + in-flight counts prevent quota overshoot.
Worker clients are reused and closed at shutdown; each request has its own saved seed and provenance.

The duplicate index covers both original splits and this batch, not other synthetic runs. Outputs are
`candidates.jsonl`, `accepted.jsonl`, `validation.jsonl`, `provenance.jsonl`, `summary.json`, and the SQLite index.
Existing output directories are refused. On a fatal error or interruption, new scheduling stops and already-running
calls are drained and saved before failure is propagated. Shutdown can take time; SDK attempts and socket timeouts are bounded.
There is no automatic resume.

After generation completes, `audit_dataset()` writes a report and linked group findings under `dataset_audit/`. Local exact/template analysis covers every record. Claude compares bounded groups of questions; approximate similarity retrieval adds likely cross-group matches. No records are deleted or certified as valid.


In [3]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def run_review_batch(output_dir):
    transport = {
        "connect_timeout": CONNECT_TIMEOUT, "read_timeout": READ_TIMEOUT,
        "sdk_max_attempts": SDK_MAX_ATTEMPTS, "retry_mode": "standard",
    }

    def factory():
        return make_client(
            MODEL_ID, connect_timeout=CONNECT_TIMEOUT,
            read_timeout=READ_TIMEOUT, sdk_max_attempts=SDK_MAX_ATTEMPTS,
        )

    def audit_factory():
        return make_client(
            AUDIT_MODEL_ID, connect_timeout=CONNECT_TIMEOUT,
            read_timeout=READ_TIMEOUT, sdk_max_attempts=SDK_MAX_ATTEMPTS,
        )

    def progress(report):
        if report.get("phase") == "dataset_audit":
            print(f"dataset audit {report['batch_id']}: {report['status']}", flush=True)
            return
        state = "accepted" if report["accepted"] else "rejected"
        print(f"{report['request_id']}: {state}", flush=True)

    return run_batch(
        output_dir, original=ORIGINAL_FILES[0], dedup_originals=ORIGINAL_FILES,
        model_id=MODEL_ID, target_per_type=TARGET_PER_TYPE,
        max_requests_per_type=MAX_REQUESTS_PER_TYPE, hop_types=HOP_TYPES_TO_REVIEW,
        concurrency=CONCURRENCY, seed=SEED, max_attempts=MAX_ATTEMPTS,
        max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
        client_factory=factory, transport_settings=transport, progress=progress,
        dataset_audit=DATASET_AUDIT, audit_client_factory=audit_factory,
        audit_model_id=AUDIT_MODEL_ID, audit_concurrency=AUDIT_CONCURRENCY,
        audit_batch_size=AUDIT_BATCH_SIZE, audit_max_prompt_chars=AUDIT_MAX_PROMPT_CHARS,
        audit_max_neighbor_batches=AUDIT_MAX_NEIGHBOR_BATCHES,
        audit_max_attempts=AUDIT_MAX_ATTEMPTS, audit_max_tokens=AUDIT_MAX_TOKENS,
    )


## Generate and automatically audit — paid calls only when enabled

Set `RUN_GENERATION = True` deliberately. Once generation completes, the dataset
redundancy audit runs automatically. Generation and audit concurrency are separate.
Existing output directories are refused. Rerun configuration for a new `RUN_DIR`.
Reset the launch switch afterward; running work does not change when configuration
text is edited. Interrupted runs are not automatically resumed.


In [4]:
RUN_GENERATION = False
if RUN_GENERATION:
    batch_summary = run_review_batch(RUN_DIR)
    print(json.dumps(batch_summary, indent=2))
else:
    print("Generation disabled. No API calls made.")


Generation disabled. No API calls made.


## Audit an existing dataset without regenerating it

Optional paid operation: set `EXISTING_RUN_DIR`, then enable the switch explicitly.
The audit output directory must be new. Generation outputs are never overwritten.


In [5]:
import boto3
from botocore.config import Config
from mas_sae.models.claude import ClaudeClient

# Refresh this profile in the terminal with: aws sso login --profile school --use-device-code
RUN_EXISTING_DATASET_AUDIT = True
# EXISTING_RUN_DIR = RUN_DIR  # Replace with the saved run directory when needed.
EXISTING_RUN_DIR = RUN_DIR  # Replace with the saved run directory when needed.
if RUN_EXISTING_DATASET_AUDIT:
    existing_audit_summary = audit_dataset(
        EXISTING_RUN_DIR / "accepted.jsonl",
        EXISTING_RUN_DIR / f"dataset_audit_{uuid4().hex[:8]}",
        model_id=AUDIT_MODEL_ID, concurrency=AUDIT_CONCURRENCY,
        batch_size=AUDIT_BATCH_SIZE, max_prompt_chars=AUDIT_MAX_PROMPT_CHARS,
        max_neighbor_batches=AUDIT_MAX_NEIGHBOR_BATCHES,
        max_attempts=AUDIT_MAX_ATTEMPTS, max_tokens=AUDIT_MAX_TOKENS,
        client_factory=lambda: ClaudeClient(
            session=boto3.Session(profile_name="school"),
            model_id=AUDIT_MODEL_ID,
            config=Config(
                connect_timeout=CONNECT_TIMEOUT,
                read_timeout=READ_TIMEOUT,
                retries={"mode": "standard", "total_max_attempts": SDK_MAX_ATTEMPTS},
            ),
        ),
    )
    print(json.dumps(existing_audit_summary, indent=2))
else:
    print("Existing-run dataset audit disabled. No API calls made.")


{
  "status": "completed_with_errors",
  "fatal_error": null,
  "secondary_errors": [],
  "audit_version": "redundancy-v1",
  "model_id": "us.anthropic.claude-opus-5",
  "concurrency": 64,
  "batch_size": 100,
  "dry_run": false,
  "max_prompt_chars": 100000,
  "max_neighbor_batches": 200,
  "max_attempts": 2,
  "max_tokens": 8192,
  "transport_settings": null,
  "input_path": "/home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/sample/20261030T173616Z_review30_8cfe1443/accepted.jsonl",
  "input_sha256": "5f07a07d6579fd3acc3685dcb54d5e7c5f12b14ab1e64be9e0d21dd16891ab73",
  "output_dir": "/home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/sample/20261030T173616Z_review30_8cfe1443/dataset_audit_9c212a50",
  "started_at": "2026-10-30T19:46:41.670031+00:00",
  "finished_at": "2026-10-30T19:50:55.551955+00:00",
  "elapsed_seconds": 253.88194561899945,
  "batches_submitted": 14,
  "batches_completed": 0,
  "batches_error": 14,
  "batches_cancelled": 0,
  "api_c

## Load saved questions and the dataset report — no API calls

Set `REVIEW_RUN_DIR` to an existing run to inspect its accepted questions and
`dataset_audit/report.md`. Audit findings are linked by record ID; they are not
per-question pass/fail approvals.


In [6]:
REVIEW_RUN_DIR = RUN_DIR  # Or: ROOT / "data/MuSiQue/synthetic/runs/<saved-run>"

if not (REVIEW_RUN_DIR / "summary.json").is_file():
    print("No saved batch here yet. Generate it explicitly or set REVIEW_RUN_DIR to an existing run.")
    records, provenance_rows, validation_rows = [], [], []
else:
    saved_summary = json.loads((REVIEW_RUN_DIR / "summary.json").read_text())
    candidate_path = REVIEW_RUN_DIR / "accepted.jsonl"
    records = read_jsonl(candidate_path) if candidate_path.exists() else []
    print(f"Displaying {len(records)} structurally accepted records from {candidate_path}")
    audit_summary_path = REVIEW_RUN_DIR / "dataset_audit/summary.json"
    if audit_summary_path.exists():
        print("Dataset audit:", json.dumps(json.loads(audit_summary_path.read_text()), indent=2))
        print("Full report:", REVIEW_RUN_DIR / "dataset_audit/report.md")
    provenance_rows = read_jsonl(REVIEW_RUN_DIR / "provenance.jsonl") if (REVIEW_RUN_DIR / "provenance.jsonl").exists() else []
    validation_rows = read_jsonl(REVIEW_RUN_DIR / "validation.jsonl") if (REVIEW_RUN_DIR / "validation.jsonl").exists() else []
    print(json.dumps(saved_summary, indent=2))
    rows = []
    for hop in HOP_TYPES_TO_REVIEW:
        requested = saved_summary["requests_by_type"].get(hop, 0)
        count = saved_summary["accepted_by_type"].get(hop, 0)
        rate = f"{count / requested:.0%}" if requested else "—"
        rows.append(f"<tr><td>{hop}</td><td>{requested}</td><td>{count}</td><td>{rate}</td></tr>")
    display(HTML("<table><tr><th>Subtype</th><th>Requests</th><th>Accepted</th>"
                 "<th>Acceptance per request</th></tr>" + "".join(rows) + "</table>"))
    failure_counts = Counter(
        error["code"] for row in validation_rows for error in row["errors"]
    )
    print("Request rejection reasons:", dict(failure_counts))
    print("A successful generation request can include an earlier rejected model response. Dataset audit findings and coverage appear separately in the report.")


Displaying 1200 structurally accepted records from /home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/sample/20261030T173616Z_review30_8cfe1443/accepted.jsonl
Dataset audit: {
  "status": "failed",
  "fatal_error": "ClientError: An error occurred (ExpiredTokenException) when calling the InvokeModel operation: The security token included in the request is expired",
  "secondary_errors": [],
  "audit_version": "redundancy-v1",
  "model_id": "us.anthropic.claude-opus-5",
  "concurrency": 4,
  "batch_size": 100,
  "dry_run": false,
  "max_prompt_chars": 100000,
  "max_neighbor_batches": 200,
  "max_attempts": 2,
  "max_tokens": 8192,
  "transport_settings": null,
  "input_path": "/home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/sample/20261030T173616Z_review30_8cfe1443/accepted.jsonl",
  "input_sha256": "5f07a07d6579fd3acc3685dcb54d5e7c5f12b14ab1e64be9e0d21dd16891ab73",
  "output_dir": "/home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/sam

Subtype,Requests,Accepted,Acceptance per request
2hop,201,200,100%
3hop1,202,200,99%
3hop2,200,200,100%
4hop1,201,200,100%
4hop2,200,200,100%
4hop3,200,200,100%


Request rejection reasons: {'duplicate': 4}
A successful generation request can include an earlier rejected model response. Dataset audit findings and coverage appear separately in the report.


## Inspect questions and documents

Change `RECORD_INDEX` to select a question (0 through `len(records) - 1`). Inspect the documents and linked dataset redundancy findings as needed.
Supporting paragraphs and step references use the saved shuffled indices. Distractors and generation history can be expanded.
Passing structural checks does not prove necessary hops or factual consistency.


In [7]:
def show_record(record_index):
    if not records:
        print("Load a saved batch first.")
        return
    record = records[record_index]
    escape = lambda value: html.escape(str(value))
    parts = [f"<h3>{record_index + 1}/{len(records)} — {escape(record['id'])}</h3>",
             f"<p><b>Question:</b> {escape(record['question'])}</p>",
             f"<p><b>Answer:</b> {escape(record['answer'])} "
             f"<b>Aliases:</b> {escape(record['answer_aliases'])}</p>", "<h4>Decomposition</h4><ol>"]
    for step in record["question_decomposition"]:
        parts.append(f"<li>{escape(step['question'])} → <b>{escape(step['answer'])}</b> "
                     f"(paragraph {step['paragraph_support_idx']})</li>")
    parts.append("</ol><h4>Supporting paragraphs</h4>")
    for paragraph in record["paragraphs"]:
        if paragraph["is_supporting"]:
            parts.append(f"<p><b>[{paragraph['idx']}] {escape(paragraph['title'])}</b><br>"
                         f"{escape(paragraph['paragraph_text'])}</p>")
    parts.append("<details><summary>Distractors</summary>")
    for paragraph in record["paragraphs"]:
        if not paragraph["is_supporting"]:
            parts.append(f"<p><b>[{paragraph['idx']}] {escape(paragraph['title'])}</b><br>"
                         f"{escape(paragraph['paragraph_text'])}</p>")
    parts.append("</details>")
    history = [row for row in provenance_rows if row.get("record_id") == record["id"]]
    parts.append("<details><summary>Generation history and usage</summary><pre>"
                 + escape(json.dumps(history, ensure_ascii=False, indent=2)) + "</pre></details>")
    audit_path = REVIEW_RUN_DIR / "dataset_audit/groups.jsonl"
    if audit_path.exists():
        groups = [row for row in read_jsonl(audit_path) if record["id"] in row["record_ids"]]
        parts.append("<details><summary>Dataset redundancy findings</summary><pre>"
                     + escape(json.dumps(groups, ensure_ascii=False, indent=2)) + "</pre></details>")
    display(HTML("".join(parts)))


RECORD_INDEX = 0
show_record(RECORD_INDEX)
